# 🏋️ Agoda Round 2 — Live Coding Problem Bank
## Format: HackerRank Live Session (interviewer watching)

> **How to use this:** Work each problem yourself first. Time yourself.
> Then reveal the solution. The goal is fluency under observation — not just correctness.

### 🎯 Live Coding Rules (burn these in)
```
1. ALWAYS narrate while typing — silence kills you in live sessions
   Say: 'I'm going to partition by hotel_id here because...'

2. State assumptions before coding, not after
   Say: 'I'll assume prices are in THB and duplicates are possible'

3. Write the skeleton first, then fill in
   Don't write perfect code top-to-bottom — show your thinking

4. When stuck, say what you're thinking
   Say: 'I'm deciding between a window function and a groupby here...'

5. Test with a tiny example before running
   Trace through 2-3 rows mentally first
```

---

## 📋 Problem Index

**SQL + Data Modeling**
- P01 ⭐⭐ — Hotel Revenue Ranking with Window Functions
- P02 ⭐⭐ — Booking Funnel Conversion Rate
- P03 ⭐⭐⭐ — Price Competitiveness Score (self-join + aggregation)
- P04 ⭐⭐⭐ — Gap Detection in Hotel Availability

**PySpark + Optimization**
- P05 ⭐⭐ — Deduplicate Kafka Events at Scale
- P06 ⭐⭐⭐ — Skew-Aware Hotel Revenue Aggregation
- P07 ⭐⭐⭐ — Spark Structured Streaming: Price Alert Pipeline
- P08 ⭐⭐⭐⭐ — Optimize a Slow Spark Job (given broken code, fix it)

**ML Feature Stores + Pipelines**
- P09 ⭐⭐ — Point-in-Time Feature Join
- P10 ⭐⭐⭐ — Feature Drift Detection
- P11 ⭐⭐⭐ — Hotel Cold-Start Feature Imputation

**Kafka + Streaming**
- P12 ⭐⭐ — Consumer Lag Monitor & Auto-scaler
- P13 ⭐⭐⭐ — Idempotent Event Processor (exactly-once)

**Monitoring + Operational Excellence**
- P14 ⭐⭐ — SLA Freshness Monitor (GoFresh pattern)
- P15 ⭐⭐⭐ — Anomaly Detection on Pipeline Metrics

**System Design Communication**
- P16 — Structured verbal answer templates (read + practice)
- P17 — Live whiteboard: Hotel Search Pipeline (timed)


---
## 🗄️ SQL + Data Modeling Problems


In [1]:
# ============================================================
# SETUP: In-memory SQLite database simulating OTA schema
# Run this cell first — all SQL problems use this data
# ============================================================
import sqlite3, pandas as pd, numpy as np
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
c = conn.cursor()

# Hotels
c.executescript('''
CREATE TABLE hotels (
    hotel_id     INTEGER PRIMARY KEY,
    name         TEXT,
    country_code TEXT,
    star_rating  INTEGER,
    city         TEXT
);
INSERT INTO hotels VALUES
  (1,  'Grand Bangkok',    'TH', 5, 'Bangkok'),
  (2,  'City Inn BKK',     'TH', 3, 'Bangkok'),
  (3,  'Phuket Pearl',     'TH', 4, 'Phuket'),
  (4,  'Tokyo Ryokan',     'JP', 4, 'Tokyo'),
  (5,  'Tokyo Business',   'JP', 3, 'Tokyo'),
  (6,  'Singapore Suites', 'SG', 5, 'Singapore'),
  (7,  'Budget SG',        'SG', 2, 'Singapore');

CREATE TABLE bookings (
    booking_id    TEXT PRIMARY KEY,
    hotel_id      INTEGER,
    customer_id   INTEGER,
    check_in      TEXT,
    check_out     TEXT,
    total_price   REAL,
    currency      TEXT,
    status        TEXT,
    created_at    TEXT
);
INSERT INTO bookings VALUES
  ('B001',1,101,'2025-06-01','2025-06-03',12000,'THB','CONFIRMED','2025-04-01'),
  ('B002',1,102,'2025-06-05','2025-06-07',12000,'THB','CONFIRMED','2025-04-02'),
  ('B003',2,103,'2025-06-01','2025-06-02', 1800,'THB','CONFIRMED','2025-04-01'),
  ('B004',2,104,'2025-06-10','2025-06-12', 3600,'THB','CANCELLED','2025-04-03'),
  ('B005',3,105,'2025-07-01','2025-07-05',28000,'THB','CONFIRMED','2025-04-05'),
  ('B006',4,106,'2025-06-15','2025-06-17',48000,'JPY','CONFIRMED','2025-04-06'),
  ('B007',4,107,'2025-06-20','2025-06-21',24000,'JPY','CONFIRMED','2025-04-07'),
  ('B008',5,108,'2025-06-01','2025-06-03',18000,'JPY','CONFIRMED','2025-04-01'),
  ('B009',6,109,'2025-06-01','2025-06-02',45000,'SGD','CONFIRMED','2025-04-02'),
  ('B010',1,110,'2025-06-20','2025-06-25',30000,'THB','CONFIRMED','2025-04-10'),
  ('B011',3,101,'2025-08-01','2025-08-03',14000,'THB','CONFIRMED','2025-04-11');

CREATE TABLE searches (
    search_id    INTEGER PRIMARY KEY,
    customer_id  INTEGER,
    hotel_id     INTEGER,
    search_ts    TEXT,
    action       TEXT   -- 'SEARCH','VIEW','INITIATE_BOOKING','CONFIRM'
);
INSERT INTO searches VALUES
  (1,101,1,'2025-04-01 10:00','SEARCH'),
  (2,101,1,'2025-04-01 10:05','VIEW'),
  (3,101,1,'2025-04-01 10:10','INITIATE_BOOKING'),
  (4,101,1,'2025-04-01 10:15','CONFIRM'),
  (5,102,1,'2025-04-02 09:00','SEARCH'),
  (6,102,1,'2025-04-02 09:10','VIEW'),
  (7,102,1,'2025-04-02 09:20','CONFIRM'),
  (8,103,2,'2025-04-01 11:00','SEARCH'),
  (9,103,2,'2025-04-01 11:10','VIEW'),
 (10,103,2,'2025-04-01 11:20','INITIATE_BOOKING'),
 (11,103,2,'2025-04-01 11:30','CONFIRM'),
 (12,104,3,'2025-04-03 14:00','SEARCH'),
 (13,104,3,'2025-04-03 14:10','VIEW'),
 (14,105,3,'2025-04-05 08:00','SEARCH'),
 (15,105,3,'2025-04-05 08:15','VIEW'),
 (16,105,3,'2025-04-05 08:30','INITIATE_BOOKING'),
 (17,105,3,'2025-04-05 08:45','CONFIRM');

CREATE TABLE price_rates (
    hotel_id      INTEGER,
    supplier_id   INTEGER,
    check_in      TEXT,
    room_type     TEXT,
    price         REAL
);
INSERT INTO price_rates VALUES
  (1,1,'2025-06-01','STANDARD',3800),(1,2,'2025-06-01','STANDARD',4000),
  (1,3,'2025-06-01','STANDARD',3700),(1,1,'2025-06-01','DELUXE',  5500),
  (2,1,'2025-06-01','STANDARD',1200),(2,2,'2025-06-01','STANDARD',1350),
  (3,1,'2025-06-01','STANDARD',2900),(3,2,'2025-06-01','STANDARD',3100),
  (3,1,'2025-06-01','DELUXE',  4200),(4,1,'2025-06-01','STANDARD',8000),
  (5,1,'2025-06-01','STANDARD',4500),(6,1,'2025-06-01','STANDARD',22000),
  (7,1,'2025-06-01','STANDARD', 800);
''')
conn.commit()
print('✅ OTA database ready. Tables: hotels, bookings, searches, price_rates')
print()
pd.read_sql('SELECT * FROM hotels', conn)


✅ OTA database ready. Tables: hotels, bookings, searches, price_rates



,hotel_id,name,country_code,star_rating,city
0,1,Grand Bangkok,TH,5,Bangkok
1,2,City Inn BKK,TH,3,Bangkok
2,3,Phuket Pearl,TH,4,Phuket
3,4,Tokyo Ryokan,JP,4,Tokyo
4,5,Tokyo Business,JP,3,Tokyo
5,6,Singapore Suites,SG,5,Singapore
6,7,Budget SG,SG,2,Singapore


---
### P01 ⭐⭐ — Hotel Revenue Ranking with Window Functions

**Problem:** For each country, rank hotels by total confirmed revenue (descending).
Show hotel name, country, total revenue, rank within country, and cumulative revenue within country.
Only include CONFIRMED bookings.

**Live coding tip:** Say *'I'll use ROW_NUMBER and SUM OVER PARTITION for this'* before writing.

**Expected columns:** `country_code, hotel_name, total_revenue, country_rank, cumulative_revenue`

⏱️ Target time: **5 minutes**


In [ ]:
# YOUR SOLUTION HERE
# Think out loud: what window functions do you need?
# Try to write it before looking at the solution below

your_query = '''
-- Write your SQL here
'''

# Uncomment to run:
# pd.read_sql(your_query, conn)


In [2]:
# ✅ SOLUTION P01
solution_p01 = '''
SELECT
    h.country_code,
    h.name                                          AS hotel_name,
    SUM(b.total_price)                              AS total_revenue,
    ROW_NUMBER() OVER (
        PARTITION BY h.country_code
        ORDER BY SUM(b.total_price) DESC
    )                                               AS country_rank,
    SUM(SUM(b.total_price)) OVER (
        PARTITION BY h.country_code
        ORDER BY SUM(b.total_price) DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )                                               AS cumulative_revenue
FROM bookings b
JOIN hotels h ON b.hotel_id = h.hotel_id
WHERE b.status = 'CONFIRMED'
GROUP BY h.country_code, h.hotel_id, h.name
ORDER BY h.country_code, country_rank
'''
result = pd.read_sql(solution_p01, conn)
print(result.to_string(index=False))
print()
print('KEY CONCEPTS:')
print('  ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...) — rank within group')
print('  Nested aggregate: SUM(SUM(...)) OVER — window on top of groupby aggregate')
print('  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW — running total')


country_code       hotel_name  total_revenue  country_rank  cumulative_revenue
          JP     Tokyo Ryokan        72000.0             1             72000.0
          JP   Tokyo Business        18000.0             2             90000.0
          SG Singapore Suites        45000.0             1             45000.0
          TH    Grand Bangkok        54000.0             1             54000.0
          TH     Phuket Pearl        42000.0             2             96000.0
          TH     City Inn BKK         1800.0             3             97800.0

KEY CONCEPTS:
  ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...) — rank within group
  Nested aggregate: SUM(SUM(...)) OVER — window on top of groupby aggregate
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW — running total


---
### P02 ⭐⭐ — Booking Funnel Conversion Rate

**Problem:** Calculate the conversion rate at each funnel step per hotel.
The funnel is: `SEARCH → VIEW → INITIATE_BOOKING → CONFIRM`.
For each hotel, show: count at each step, SEARCH→CONFIRM overall conversion rate.

**Live coding tip:** Think PIVOT or conditional aggregation (`COUNT + CASE WHEN`).

⏱️ Target: **6 minutes**


In [ ]:
# YOUR SOLUTION HERE
your_query_p02 = '''
-- Write your SQL here
'''
# pd.read_sql(your_query_p02, conn)


In [ ]:
# ✅ SOLUTION P02
solution_p02 = '''
SELECT
    h.name                                              AS hotel_name,
    COUNT(CASE WHEN s.action = 'SEARCH'           THEN 1 END) AS searches,
    COUNT(CASE WHEN s.action = 'VIEW'             THEN 1 END) AS views,
    COUNT(CASE WHEN s.action = 'INITIATE_BOOKING' THEN 1 END) AS initiated,
    COUNT(CASE WHEN s.action = 'CONFIRM'          THEN 1 END) AS confirmed,
    ROUND(
        100.0 * COUNT(CASE WHEN s.action = 'CONFIRM' THEN 1 END)
              / NULLIF(COUNT(CASE WHEN s.action = 'SEARCH' THEN 1 END), 0),
        1
    )                                                   AS conversion_pct
FROM searches s
JOIN hotels h ON s.hotel_id = h.hotel_id
GROUP BY h.hotel_id, h.name
ORDER BY conversion_pct DESC
'''
result = pd.read_sql(solution_p02, conn)
print(result.to_string(index=False))
print()
print('KEY CONCEPTS:')
print('  CASE WHEN inside COUNT/SUM = conditional aggregation (the SQL pivot trick)')
print('  NULLIF(x, 0) prevents division-by-zero — always use this in conversion rate queries')
print('  ROUND(..., 1) for clean percentages')


---
### P03 ⭐⭐⭐ — Price Competitiveness Score

**Problem:** For each hotel on '2025-06-01', compute its price competitiveness score:
`competitiveness = hotel_min_price / city_avg_min_price`
A score < 1.0 means the hotel is cheaper than city average (competitive).
Show: hotel name, city, star_rating, hotel_min_price, city_avg_min_price, competitiveness_score.
Sort by most competitive first.

**Live coding tip:** This needs two levels — first aggregate per hotel, then window over city.

⏱️ Target: **8 minutes**


In [ ]:
# YOUR SOLUTION HERE
your_query_p03 = '''
-- Hint: CTE for hotel min prices, then window for city avg
'''
# pd.read_sql(your_query_p03, conn)


In [ ]:
# ✅ SOLUTION P03
solution_p03 = '''
WITH hotel_min_prices AS (
    SELECT
        pr.hotel_id,
        MIN(pr.price) AS min_price
    FROM price_rates pr
    WHERE pr.check_in = '2025-06-01'
      AND pr.room_type = 'STANDARD'
    GROUP BY pr.hotel_id
),
hotel_with_meta AS (
    SELECT
        h.hotel_id,
        h.name,
        h.city,
        h.star_rating,
        hmp.min_price
    FROM hotel_min_prices hmp
    JOIN hotels h ON hmp.hotel_id = h.hotel_id
)
SELECT
    name,
    city,
    star_rating,
    min_price                   AS hotel_min_price,
    ROUND(AVG(min_price) OVER (PARTITION BY city), 0) AS city_avg_min_price,
    ROUND(
        min_price / AVG(min_price) OVER (PARTITION BY city),
        3
    )                           AS competitiveness_score
FROM hotel_with_meta
ORDER BY competitiveness_score ASC
'''
result = pd.read_sql(solution_p03, conn)
print(result.to_string(index=False))
print()
print('KEY CONCEPTS:')
print('  Two-level aggregation: GROUP BY (per hotel), then WINDOW (per city)')
print('  CTE chain keeps logic readable — interviewers love this')
print('  AVG() OVER (PARTITION BY city) = city average without collapsing rows')


---
### P04 ⭐⭐⭐ — Gap Detection in Hotel Availability

**Problem:** Given a bookings table, find all hotels that have a GAP of 3+ days
with NO confirmed bookings between 2025-06-01 and 2025-07-31.
A 'gap' means the hotel has bookings in that window but also has 3+ consecutive
unbooked days between those bookings.

**Why this matters:** Agoda uses this to identify hotels to upsell promotions to.

**Live coding tip:** Use LAG/LEAD window functions to find the gap between consecutive bookings.

⏱️ Target: **10 minutes** — this is a hard one


In [ ]:
# ✅ SOLUTION P04 — Gap Detection with LAG
solution_p04 = '''
WITH hotel_stays AS (
    SELECT
        hotel_id,
        check_in,
        check_out,
        LAG(check_out) OVER (
            PARTITION BY hotel_id
            ORDER BY check_in
        ) AS prev_check_out
    FROM bookings
    WHERE status = 'CONFIRMED'
      AND check_in  >= '2025-06-01'
      AND check_out <= '2025-07-31'
),
gaps AS (
    SELECT
        hotel_id,
        prev_check_out              AS gap_start,
        check_in                    AS gap_end,
        julianday(check_in) - julianday(prev_check_out) AS gap_days
    FROM hotel_stays
    WHERE prev_check_out IS NOT NULL   -- skip first booking (no previous)
)
SELECT
    h.name,
    g.gap_start,
    g.gap_end,
    CAST(g.gap_days AS INTEGER) AS gap_days
FROM gaps g
JOIN hotels h ON g.hotel_id = h.hotel_id
WHERE g.gap_days >= 3
ORDER BY g.gap_days DESC
'''
result = pd.read_sql(solution_p04, conn)
print(result.to_string(index=False))
print()
print('KEY CONCEPTS:')
print('  LAG(col) OVER (PARTITION BY hotel_id ORDER BY check_in) = previous row value')
print('  julianday() in SQLite = date difference in days (use DATEDIFF in MySQL/BigQuery)')
print('  WHERE prev_check_out IS NOT NULL filters the first booking per hotel')
print('  Pattern: LAG/LEAD for sequential gap problems, always partition by entity')


---
## ⚡ PySpark + Optimization Problems


In [ ]:
# ============================================================
# PySpark Setup (pure Python simulation — same logic, portable)
# In real interview: use SparkSession directly
# ============================================================
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

random.seed(42)
np.random.seed(42)

# Simulate large-scale OTA event data
N = 100_000  # simulate 100K events

hotel_ids = np.random.choice(
    [1, 1, 1, 2, 2, 3, 4, 5, 6, 7, 8, 9, 10],  # hotel 1 is 'hot' — appears 3x more
    size=N
)

events_df = pd.DataFrame({
    'event_id':      [f'EVT_{i:08d}' for i in range(N)],
    'hotel_id':      hotel_ids,
    'user_id':       np.random.randint(1000, 9999, N),
    'event_type':    np.random.choice(['SEARCH','VIEW','BOOK'], N, p=[0.7,0.2,0.1]),
    'price':         np.abs(np.random.normal(3000, 800, N)).round(2),
    'revenue':       np.abs(np.random.normal(2700, 700, N)).round(2),
    'event_ts':      pd.date_range('2025-04-01', periods=N, freq='1s'),
    'country':       np.random.choice(['TH','JP','SG','US','AU'], N, p=[0.4,0.2,0.2,0.1,0.1]),
})

# Inject ~5% duplicates (simulating at-least-once Kafka delivery)
dup_idx = np.random.choice(N, size=int(N*0.05), replace=False)
duplicates = events_df.iloc[dup_idx].copy()
events_df = pd.concat([events_df, duplicates], ignore_index=True).sample(frac=1, random_state=42)

print(f'Dataset: {len(events_df):,} rows (including ~5% duplicates)')
print(f'Hot hotel distribution:')
print(events_df['hotel_id'].value_counts().head(5))
print(f'\nColumns: {list(events_df.columns)}')


---
### P05 ⭐⭐ — Deduplicate Kafka Events at Scale

**Problem:** The events DataFrame has ~5% duplicate rows (same `event_id`).
Deduplicate it keeping only the FIRST occurrence of each `event_id`.
Show: row count before and after, and verify no duplicate event_ids remain.

**In Spark this would use:** `dropDuplicates` or window `ROW_NUMBER`.
Here use pandas equivalents (same logic, same interview answer).

**Live coding tip:** Mention both approaches — `dropDuplicates(['event_id'])` (simple)
vs `ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY event_ts)` (when you need control
over WHICH duplicate to keep).

⏱️ Target: **4 minutes**


In [ ]:
# YOUR SOLUTION HERE
# Approach 1: simple drop_duplicates
# Approach 2: sort + keep first (when you need most-recent or oldest)

# deduped = ...
# print(f'Before: {len(events_df):,} | After: {len(deduped):,}')


In [ ]:
# ✅ SOLUTION P05
print('=== Approach 1: Simple dedup (keep any occurrence) ===')
deduped_simple = events_df.drop_duplicates(subset=['event_id'])
print(f'Before: {len(events_df):,} | After: {len(deduped_simple):,}')
print(f'Duplicate event_ids remaining: {deduped_simple["event_id"].duplicated().sum()}')

print()
print('=== Approach 2: Keep EARLIEST occurrence (deterministic, preferred in prod) ===')
deduped_earliest = (
    events_df
    .sort_values('event_ts')                        # sort by timestamp
    .drop_duplicates(subset=['event_id'], keep='first')  # keep earliest
    .reset_index(drop=True)
)
print(f'Before: {len(events_df):,} | After: {len(deduped_earliest):,}')

print()
print('=== In Spark (what you say in the interview) ===')
spark_code = '''
# Simple:
deduped = df.dropDuplicates(['event_id'])

# Controlled (keep earliest):
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy('event_id').orderBy('event_ts')
deduped = (
    df.withColumn('rn', F.row_number().over(w))
      .filter('rn = 1')
      .drop('rn')
)
# Why prefer this over dropDuplicates?
# dropDuplicates picks non-deterministically in distributed setting.
# ROW_NUMBER with ORDER BY gives you a guaranteed deterministic result.
'''
print(spark_code)


---
### P06 ⭐⭐⭐ — Skew-Aware Hotel Revenue Aggregation

**Problem:** Aggregate total revenue per hotel. BUT hotel_id=1 has ~3x more events
than any other hotel (simulating Bangkok's top hotel).

Tasks:
1. Detect the skew (show distribution)
2. Compute revenue per hotel correctly
3. Explain how you'd handle this in Spark with AQE + salting

**This is a verbal + code combo question — the explanation matters as much as the code.**

⏱️ Target: **8 minutes**


In [ ]:
# ✅ SOLUTION P06 — Skew Detection + Aggregation
print('=== Step 1: Detect Skew ===')
hotel_counts = deduped_earliest['hotel_id'].value_counts()
max_count = hotel_counts.max()
mean_count = hotel_counts.mean()
skew_ratio = max_count / mean_count
print(f'Event counts per hotel (top 5):')
print(hotel_counts.head(5))
print(f'\nSkew ratio (max/mean): {skew_ratio:.1f}x  <- if > 5x, a problem in Spark')

print()
print('=== Step 2: Revenue aggregation (correct result) ===')
revenue = (
    deduped_earliest[deduped_earliest['event_type'] == 'BOOK']
    .groupby('hotel_id')['revenue']
    .agg(['sum','count','mean'])
    .rename(columns={'sum':'total_revenue','count':'booking_count','mean':'avg_revenue'})
    .sort_values('total_revenue', ascending=False)
)
print(revenue.round(0))

print()
print('=== Step 3: Spark Skew Handling (what to say) ===')
explanation = '''
OPTION A — Spark AQE (Spark 3.0+, recommended first):
  spark.conf.set('spark.sql.adaptive.enabled', 'true')
  spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
  AQE auto-detects partitions > skewedPartitionFactor * median size
  and splits them. Zero code change needed.

OPTION B — Manual Salting (when AQE isn't enough):
  SALT_FACTOR = 20
  df_salted = df.withColumn(
      'salted_id',
      F.concat(F.col('hotel_id'), F.lit('_'), (F.rand()*SALT_FACTOR).cast('int'))
  )
  partial = df_salted.groupBy('salted_id','hotel_id').agg(F.sum('revenue'))
  final   = partial.groupBy('hotel_id').agg(F.sum('partial_sum'))
  -- Two-phase aggregation: distribute hot key -> merge results

OPTION C — Broadcast join for dimension tables:
  If hotel metadata is small (<1GB), broadcast it:
  events.join(F.broadcast(hotel_meta), 'hotel_id')
  Eliminates shuffle entirely for the join side.
'''
print(explanation)


---
### P07 ⭐⭐⭐ — Spark Streaming: Price Alert Pipeline

**Problem:** Write a Spark Structured Streaming pipeline that:
1. Reads price events from Kafka topic `hotel.prices`
2. Computes 10-minute rolling minimum price per hotel
3. Alerts when a hotel's current min price drops >20% below its rolling average

**In this cell:** write the full pipeline skeleton with correct config.
Narrate every config choice as if the interviewer is watching.

⏱️ Target: **10 minutes** — focus on structure, not perfection


In [ ]:
# ✅ SOLUTION P07 — Streaming Price Alert
streaming_pipeline = '''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = SparkSession.builder\\
    .appName("PriceAlertPipeline")\\
    .config("spark.sql.streaming.checkpointLocation", "hdfs:///checkpoints/price-alert/")\\
    .getOrCreate()

# Step 1: Define schema — ALWAYS explicit in streaming (never infer)
schema = StructType([
    StructField("hotel_id",       StringType()),
    StructField("price",          DoubleType()),
    StructField("room_type",      StringType()),
    StructField("event_timestamp", LongType()),   # Unix millis
])

# Step 2: Read from Kafka
raw_stream = spark.readStream\\
    .format("kafka")\\
    .option("kafka.bootstrap.servers", "kafka-analytics:9092")\\
    .option("subscribe", "hotel.prices")\\
    .option("startingOffsets", "latest")\\
    .option("maxOffsetsPerTrigger", 500_000)  \\
    .load()

# Step 3: Parse JSON
parsed = raw_stream\\
    .select(F.from_json(F.col("value").cast("string"), schema).alias("d"))\\
    .select("d.*")\\
    .withColumn("event_time",
                F.from_unixtime(F.col("event_timestamp") / 1000).cast("timestamp"))

# Step 4: Windowed aggregation with watermark
# Watermark: how late can data arrive?
# Price events have ~10s p99 latency (2-step logging). Use 2-min watermark for safety.
windowed = parsed\\
    .withWatermark("event_time", "2 minutes")\\
    .groupBy(
        "hotel_id",
        F.window("event_time", "10 minutes", "5 minutes")  # sliding: 10min window, 5min slide
    )\\
    .agg(
        F.min("price").alias("window_min_price"),
        F.avg("price").alias("window_avg_price"),
        F.count("*").alias("event_count")
    )

# Step 5: Alert when price drops >20% below rolling average
alerts = windowed.filter(
    F.col("window_min_price") < F.col("window_avg_price") * 0.80
)

# Step 6: Write alerts to Kafka topic for downstream action
query = alerts.selectExpr(
    "hotel_id AS key",
    "to_json(struct(*)) AS value"
).writeStream\\
    .format("kafka")\\
    .option("kafka.bootstrap.servers", "kafka-analytics:9092")\\
    .option("topic", "hotel.price-alerts")\\
    .option("checkpointLocation", "hdfs:///checkpoints/price-alert/")\\
    .outputMode("append")\\
    .trigger(processingTime="30 seconds")\\
    .start()

query.awaitTermination()
'''

print(streaming_pipeline)
print()
print('NARRATION POINTS (say these out loud to interviewer):')
narration = [
    'I define the schema explicitly because inferring schema triggers a batch job on streaming',
    'maxOffsetsPerTrigger = 500K prevents OOM if there is a backlog spike',
    'The watermark tells Spark how long to wait for late data before closing a window',
    'I use a sliding window (not tumbling) so a price drop is caught within 5 min',
    'outputMode=append because watermark allows us to emit completed windows only',
    'checkpointLocation is critical — without it, job restart = reprocess from startingOffsets',
    'Writing alerts back to Kafka decouples the alert consumer from this pipeline',
]
for i, point in enumerate(narration, 1):
    print(f'  {i}. {point}')


---
### P08 ⭐⭐⭐⭐ — Fix the Broken Spark Job

**Problem:** The code below is a real ETL job with **5 bugs / anti-patterns**.
Find them, explain why each is wrong, and write the fixed version.

**This type of question is very common in live sessions — shows you know production Spark.**

⏱️ Target: **12 minutes**


In [ ]:
# BROKEN CODE — find 5 issues
broken_code = '''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("BrokenETL").getOrCreate()

# (1) Read hotel prices
prices = spark.read.option("inferSchema", "true").csv("hdfs:///data/prices/")

# (2) Filter for Thailand
th_prices = prices.filter("country = 'TH'")

# (3) Check how many records we have
print(f"TH price records: {th_prices.count()}")

# (4) Join with hotel metadata (3.6M hotels, 200 bytes each)
hotel_meta = spark.read.parquet("hdfs:///data/hotel_metadata/")
enriched = th_prices.join(hotel_meta, "hotel_id")

# (5) Aggregate revenue per hotel
result = enriched.groupBy("hotel_id").agg(F.sum("price").alias("total"))

# (6) Write output
result.write.mode("overwrite").csv("hdfs:///output/th_revenue/")
'''
print(broken_code)
print()
print('Find the 5 bugs/anti-patterns before scrolling to the solution...')


In [ ]:
# ✅ SOLUTION P08 — Annotated Fixes
bugs = [
    {
        'bug': 'inferSchema=True on CSV triggers a full scan of the dataset',
        'risk': 'For TB-scale data: doubles read time, wrong types possible',
        'fix': 'Always define schema explicitly: spark.read.schema(defined_schema).csv(...)'
    },
    {
        'bug': 'th_prices.count() mid-pipeline triggers a full Spark action',
        'risk': 'Forces complete materialization of th_prices — wasted work if you need it downstream',
        'fix': 'Remove or move to after .cache(). Use Spark UI for counts, not inline count()'
    },
    {
        'bug': 'th_prices is computed twice: once for count(), once for join()',
        'risk': 'Full re-scan of prices CSV for each action — at TB scale this doubles runtime',
        'fix': 'th_prices.cache() after filtering, before the count and join'
    },
    {
        'bug': 'join(hotel_meta, hotel_id) without broadcast hint',
        'risk': 'hotel_meta (3.6M * 200B = ~720MB) triggers shuffle join — expensive',
        'fix': 'enriched = th_prices.join(F.broadcast(hotel_meta), "hotel_id")'
    },
    {
        'bug': 'Write output as CSV',
        'risk': 'CSV: no compression, no predicate pushdown, slow reads for downstream',
        'fix': 'result.write.mode("overwrite").parquet("hdfs:///output/th_revenue/")'
    },
]

for i, bug in enumerate(bugs, 1):
    print(f'BUG {i}: {bug["bug"]}')
    print(f'  Risk: {bug["risk"]}')
    print(f'  Fix:  {bug["fix"]}')
    print()

print('=== FIXED VERSION ===')
fixed = '''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = SparkSession.builder\\
    .appName("FixedETL")\\
    .config("spark.sql.adaptive.enabled", "true")\\
    .getOrCreate()

# FIX 1: Explicit schema
price_schema = StructType([
    StructField("hotel_id",   StringType()),
    StructField("country",    StringType()),
    StructField("price",      DoubleType()),
    StructField("check_in",   StringType()),
])
prices = spark.read.schema(price_schema).parquet("hdfs:///data/prices/")

# FIX 3: Cache before reuse
th_prices = prices.filter("country = 'TH'").cache()

# FIX 2: count() only if truly needed, after cache
# print(f'TH records: {th_prices.count()}')  # now only one scan

# FIX 4: Broadcast small dimension table
hotel_meta = spark.read.parquet("hdfs:///data/hotel_metadata/")
enriched = th_prices.join(F.broadcast(hotel_meta), "hotel_id")

result = enriched.groupBy("hotel_id").agg(F.sum("price").alias("total"))

# FIX 5: Write Parquet, not CSV
result.write.mode("overwrite").parquet("hdfs:///output/th_revenue/")
'''
print(fixed)


---
## 🧠 ML Feature Store + Pipelines Problems


---
### P09 ⭐⭐ — Point-in-Time Feature Join

**Problem:** Given booking events (with timestamps) and daily hotel feature snapshots,
join each booking to the feature snapshot that was available AT THE TIME of booking
(not the latest). This prevents data leakage.

⏱️ Target: **8 minutes**


In [ ]:
# ✅ SOLUTION P09 — PIT Join
import pandas as pd

# Booking labels
bookings = pd.DataFrame({
    'booking_id':  ['B001','B002','B003','B004'],
    'hotel_id':    [1, 1, 2, 2],
    'booked_at':   pd.to_datetime(['2025-01-20','2025-03-10','2025-02-05','2025-04-01']),
    'converted':   [1, 1, 0, 1],
})

# Feature snapshots (hotel review count, updated daily)
features = pd.DataFrame({
    'hotel_id':      [1,  1,  1,  2,  2,  2],
    'snapshot_date': pd.to_datetime(['2025-01-01','2025-02-01','2025-04-01',
                                     '2025-01-01','2025-03-01','2025-04-01']),
    'review_count':  [50, 55, 70, 200, 210, 230],
    'avg_score':     [4.2,4.3,4.5, 4.7, 4.7, 4.8],
})

print('Bookings (labels):')
print(bookings)
print('\nFeature snapshots:')
print(features)

# PIT Join: for each booking, get the LATEST feature snapshot BEFORE booked_at
def pit_join(labels, feats, entity_col, label_ts, feat_ts):
    rows = []
    for _, label in labels.iterrows():
        valid = feats[
            (feats[entity_col] == label[entity_col]) &
            (feats[feat_ts] <= label[label_ts])
        ].sort_values(feat_ts)
        if not valid.empty:
            latest = valid.iloc[-1]
            row = {**label.to_dict(),
                   'review_count': latest['review_count'],
                   'avg_score':    latest['avg_score'],
                   'feature_date': latest[feat_ts]}
            rows.append(row)
    return pd.DataFrame(rows)

result = pit_join(bookings, features, 'hotel_id', 'booked_at', 'snapshot_date')
print('\n=== PIT Join Result (no data leakage) ===')
print(result[['booking_id','hotel_id','booked_at','feature_date','review_count','avg_score','converted']].to_string(index=False))
print()
print('NOTE: B001 (2025-01-20) uses snapshot 2025-01-01 (review=50), not the latest 70')
print('Using latest (70) would be DATA LEAKAGE — model knows the future.')
print()
print('In Spark (production): use Feathr / Feast as-of join, or ASOF JOIN in StarRocks')


---
### P10 ⭐⭐⭐ — Feature Drift Detection

**Problem:** You have hotel features computed today and 30 days ago.
Detect which features have drifted significantly (Z-score > 3 or PSI > 0.2).

**Why this matters:** Feature drift → model performance degradation before you catch it.

⏱️ Target: **10 minutes**


In [ ]:
# ✅ SOLUTION P10 — Feature Drift Detection
import numpy as np
from scipy import stats

np.random.seed(42)
n = 5000

# Simulate feature distributions: baseline (30d ago) vs current
baseline = pd.DataFrame({
    'review_score':     np.random.normal(4.2, 0.4, n),
    'price_usd':        np.abs(np.random.normal(120, 40, n)),
    'booking_rate_7d':  np.random.beta(2, 5, n),
    'photo_score':      np.random.uniform(0.5, 1.0, n),
})

current = pd.DataFrame({
    'review_score':     np.random.normal(4.2, 0.4, n),        # stable
    'price_usd':        np.abs(np.random.normal(150, 40, n)), # DRIFTED (mean up 25%)
    'booking_rate_7d':  np.random.beta(2, 5, n),              # stable
    'photo_score':      np.random.uniform(0.3, 0.9, n),       # DRIFTED (shifted down)
})

def psi(baseline_vals, current_vals, buckets=10):
    '''Population Stability Index — standard feature drift metric.
    PSI < 0.1: no drift. 0.1-0.2: minor drift. > 0.2: major drift.'''
    lo = min(baseline_vals.min(), current_vals.min())
    hi = max(baseline_vals.max(), current_vals.max())
    bins = np.linspace(lo, hi, buckets + 1)

    base_pct = np.histogram(baseline_vals, bins)[0] / len(baseline_vals)
    curr_pct = np.histogram(current_vals,  bins)[0] / len(current_vals)

    # Avoid log(0)
    base_pct = np.where(base_pct == 0, 0.0001, base_pct)
    curr_pct = np.where(curr_pct == 0, 0.0001, curr_pct)

    return np.sum((curr_pct - base_pct) * np.log(curr_pct / base_pct))

print(f'{'Feature':<20} {'Mean_Base':>10} {'Mean_Curr':>10} {'PSI':>8} {'Status':>10}')
print('-' * 62)
for col in baseline.columns:
    psi_val = psi(baseline[col].values, current[col].values)
    mean_b  = baseline[col].mean()
    mean_c  = current[col].mean()
    status  = '🔴 DRIFTED' if psi_val > 0.2 else ('🟡 MINOR' if psi_val > 0.1 else '✅ STABLE')
    print(f'{col:<20} {mean_b:>10.3f} {mean_c:>10.3f} {psi_val:>8.3f} {status:>10}')

print()
print('ACTION when PSI > 0.2:')
print('  1. Alert ML platform team immediately')
print('  2. Investigate root cause (data pipeline bug vs. real-world change?)')
print('  3. Retrain model with recent data if real-world distribution changed')
print('  4. Add upstream data quality check for the drifted feature')


---
### P11 ⭐⭐⭐ — Cold-Start Feature Imputation for New Hotels

**Problem:** New hotels have no historical data (no reviews, no booking rate).
Design and implement a feature imputation strategy so the ML model can still score them.

**Strategy tiers:**
1. City + star_rating peers average
2. City average (if not enough peers)
3. Global average (last resort)

⏱️ Target: **8 minutes**


In [ ]:
# ✅ SOLUTION P11 — Hierarchical Cold-Start Imputation
hotels_with_features = pd.DataFrame({
    'hotel_id':       [1,2,3,4,5,6,7,8],
    'city':           ['Bangkok','Bangkok','Phuket','Bangkok','Tokyo','Tokyo','Singapore','Singapore'],
    'star_rating':    [5, 3, 4, 3, 4, 3, 5, 2],
    'review_count':   [500, 120, None, 80, None, 200, 350, None],  # None = new hotel
    'booking_rate':   [0.12, 0.08, None, 0.09, None, 0.07, 0.15, None],
})

def hierarchical_impute(df, feature_col, group_cols_hierarchy):
    '''
    Impute nulls using hierarchy:
    Level 1: group by all group_cols
    Level 2: group by first n-1 cols
    Level 3: global mean
    '''
    result = df.copy()
    result['imputation_level'] = 'actual'

    for level, n_cols in enumerate(range(len(group_cols_hierarchy), 0, -1), start=1):
        group_cols = group_cols_hierarchy[:n_cols]
        level_means = df.groupby(group_cols)[feature_col].mean().reset_index()
        level_means = level_means.rename(columns={feature_col: f'_imputed_{level}'})

        result = result.merge(level_means, on=group_cols, how='left')
        mask = result[feature_col].isna() & result[f'_imputed_{level}'].notna()
        result.loc[mask, feature_col] = result.loc[mask, f'_imputed_{level}']
        result.loc[mask, 'imputation_level'] = f'level_{level}_{"_".join(group_cols)}'
        result = result.drop(columns=[f'_imputed_{level}'])

    # Final fallback: global mean
    global_mean = df[feature_col].mean()
    still_null = result[feature_col].isna()
    result.loc[still_null, feature_col] = global_mean
    result.loc[still_null, 'imputation_level'] = 'global_mean'

    return result

imputed = hierarchical_impute(
    hotels_with_features.copy(),
    feature_col='review_count',
    group_cols_hierarchy=['city','star_rating']  # try city+star first, then city only
)

print('Result (review_count imputed for new hotels):')
print(imputed[['hotel_id','city','star_rating','review_count','imputation_level']].to_string(index=False))
print()
print('KEY INSIGHT: Always track imputation_level as a feature itself!')
print('  The model can learn that imputed hotels are less certain -> lower confidence score')
print('  Never silently impute without logging HOW you imputed')


---
## 📨 Kafka + Streaming Problems


---
### P12 ⭐⭐ — Consumer Lag Monitor & Auto-scaler

**Problem:** Implement a Kafka consumer lag monitor that:
1. Tracks lag per partition
2. Alerts when lag exceeds a threshold for > N minutes
3. Recommends scaling actions

⏱️ Target: **8 minutes**


In [ ]:
# ✅ SOLUTION P12 — Kafka Lag Monitor
from dataclasses import dataclass, field
from typing import Dict, List
from datetime import datetime, timedelta
import time

@dataclass
class PartitionLag:
    topic: str
    partition: int
    latest_offset: int
    committed_offset: int
    timestamp: datetime = field(default_factory=datetime.utcnow)

    @property
    def lag(self) -> int:
        return self.latest_offset - self.committed_offset


class KafkaLagMonitor:
    def __init__(self, lag_threshold: int = 100_000, alert_duration_min: int = 5):
        self.lag_threshold = lag_threshold
        self.alert_duration = timedelta(minutes=alert_duration_min)
        self.lag_history: Dict[tuple, List[PartitionLag]] = {}
        self.alerts_sent = set()

    def record(self, lag: PartitionLag):
        key = (lag.topic, lag.partition)
        if key not in self.lag_history:
            self.lag_history[key] = []
        self.lag_history[key].append(lag)
        # Keep last 30 readings
        self.lag_history[key] = self.lag_history[key][-30:]
        self._evaluate(key)

    def _evaluate(self, key):
        history = self.lag_history[key]
        if len(history) < 2:
            return

        # Check if lag has been above threshold for alert_duration
        above_threshold = [h for h in history if h.lag > self.lag_threshold]
        if not above_threshold:
            self.alerts_sent.discard(key)  # reset alert state
            return

        duration = above_threshold[-1].timestamp - above_threshold[0].timestamp
        if duration >= self.alert_duration and key not in self.alerts_sent:
            self._alert(key, above_threshold[-1])
            self.alerts_sent.add(key)

    def _alert(self, key, latest: PartitionLag):
        topic, partition = key
        lag_rate = self._compute_lag_rate(key)
        action = self._recommend_action(latest.lag, lag_rate)
        print(f'🔴 ALERT: {topic}[{partition}] lag={latest.lag:,} for >{self.alert_duration.seconds//60}min')
        print(f'   Lag growth rate: {lag_rate:+,.0f} events/min')
        print(f'   Recommended action: {action}')

    def _compute_lag_rate(self, key) -> float:
        history = self.lag_history[key]
        if len(history) < 2:
            return 0.0
        dt = (history[-1].timestamp - history[-2].timestamp).total_seconds() / 60
        dl = history[-1].lag - history[-2].lag
        return dl / dt if dt > 0 else 0

    def _recommend_action(self, current_lag: int, lag_rate: float) -> str:
        if lag_rate > 10_000:
            return 'URGENT: Add 2x more consumer replicas immediately'
        elif lag_rate > 1_000:
            return 'Add 1 consumer replica, check for slow processing bottleneck'
        elif current_lag > 1_000_000:
            return 'Backlog large but not growing — monitor, may self-resolve'
        return 'Monitor: lag threshold exceeded but rate stable'


# Demo
monitor = KafkaLagMonitor(lag_threshold=50_000, alert_duration_min=2)

# Simulate lag growing over time on partition 1
base_time = datetime(2025, 4, 17, 10, 0, 0)
for i in range(10):
    ts = base_time + timedelta(minutes=i)
    # Partition 0: healthy
    monitor.record(PartitionLag('hotel.prices', 0, 1_000_000 + i*1000, 1_000_000 + i*900, ts))
    # Partition 1: lagging fast
    monitor.record(PartitionLag('hotel.prices', 1, 1_000_000 + i*50_000, 900_000, ts))

print('\n=== Summary ===')
for key, history in monitor.lag_history.items():
    latest = history[-1]
    print(f'  {key[0]}[{key[1]}]: current lag = {latest.lag:>8,}')


---
### P13 ⭐⭐⭐ — Idempotent Event Processor (Exactly-Once)

**Problem:** Build an exactly-once booking event processor.
If the same booking_request_id is submitted twice (network retry), the second
request should return the original result without processing again.

⏱️ Target: **10 minutes**


In [ ]:
# ✅ SOLUTION P13 — Idempotent Processor
import hashlib, json
from datetime import datetime, timedelta

class IdempotentBookingProcessor:
    '''
    Exactly-once booking processing via idempotency keys.
    In production: Redis instead of dict, with 24h TTL.
    '''

    def __init__(self, ttl_hours: int = 24):
        self._processed: dict = {}  # idempotency_key -> result
        self._ttl = timedelta(hours=ttl_hours)
        self._process_count = 0   # track how many times we actually processed

    def process_booking(self, request: dict) -> dict:
        idem_key = request.get('idempotency_key')
        if not idem_key:
            raise ValueError('idempotency_key is required for booking requests')

        # Check if already processed (cache hit)
        if idem_key in self._processed:
            stored = self._processed[idem_key]
            if datetime.utcnow() - stored['processed_at'] < self._ttl:
                print(f'  [IDEMPOTENT] Returning cached result for {idem_key}')
                return {**stored['result'], 'idempotent': True}

        # Process for the first time
        self._process_count += 1
        result = self._do_process(request)

        # Store result
        self._processed[idem_key] = {
            'result': result,
            'processed_at': datetime.utcnow()
        }
        return result

    def _do_process(self, request: dict) -> dict:
        '''Simulate actual booking creation.'''
        booking_id = hashlib.md5(json.dumps(request, sort_keys=True).encode()).hexdigest()[:8]
        print(f'  [PROCESSING] Creating booking {booking_id} for hotel {request["hotel_id"]}')
        return {
            'booking_id': f'BK-{booking_id}',
            'hotel_id':   request['hotel_id'],
            'status':     'CONFIRMED',
            'created_at': datetime.utcnow().isoformat(),
            'idempotent': False
        }


processor = IdempotentBookingProcessor()

# First attempt
req = {'idempotency_key': 'USER-101-SESS-XYZ-001', 'hotel_id': 12345, 'room_type': 'DELUXE'}
print('Attempt 1 (original):')
r1 = processor.process_booking(req)
print(f'  Result: {r1}')

# Retry (network timeout)
print('\nAttempt 2 (retry — user clicked again):')
r2 = processor.process_booking(req)
print(f'  Result: {r2}')

# Different booking
print('\nAttempt 3 (different booking, different key):')
req2 = {'idempotency_key': 'USER-102-SESS-ABC-001', 'hotel_id': 99999, 'room_type': 'STANDARD'}
r3 = processor.process_booking(req2)
print(f'  Result: {r3}')

print(f'\nTotal actual processings: {processor._process_count} (of 3 requests)')
print('Same booking_id returned for both attempts 1 and 2:', r1["booking_id"] == r2["booking_id"])
print()
print('PRODUCTION NOTES:')
print('  - Use Redis SETNX (atomic set-if-not-exists) instead of Python dict')
print('  - SET idempotency_key result EX 86400  (24h TTL)')
print('  - Combine with Kafka transactional.id for end-to-end exactly-once')


---
## 📡 Monitoring + Operational Excellence Problems


---
### P14 ⭐⭐ — SLA Freshness Monitor (GoFresh Pattern)

**Problem:** Build a data freshness monitor for Agoda's pipeline tables.
Each table has an SLA (max minutes since last update).
Alert and classify severity when the SLA is breached.

⏱️ Target: **7 minutes**


In [ ]:
# ✅ SOLUTION P14 — GoFresh Pattern
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Optional

@dataclass
class TableSLA:
    name: str
    max_lag_minutes: int
    severity: str        # 'CRITICAL', 'ERROR', 'WARNING'
    owner_team: str

@dataclass
class FreshnessResult:
    table: str
    last_updated: datetime
    lag_minutes: float
    sla_minutes: int
    breached: bool
    severity: str

class GoFreshMonitor:
    SLA_CONFIG = [
        TableSLA('hotel.prices.streaming',    10,  'CRITICAL', 'pricing-team'),
        TableSLA('booking_events',             5,  'CRITICAL', 'platform-team'),
        TableSLA('financial_unified_daily',   90,  'CRITICAL', 'finance-team'),
        TableSLA('hotel_features_daily',     120,  'ERROR',    'ml-platform'),
        TableSLA('ml_training_dataset',    1440,   'WARNING',  'ml-platform'),
    ]

    def check_all(self, table_last_updated: dict) -> list:
        now = datetime.utcnow()
        results = []
        for sla in self.SLA_CONFIG:
            last_upd = table_last_updated.get(sla.name)
            if last_upd is None:
                lag_min = float('inf')
                breached = True
            else:
                lag_min = (now - last_upd).total_seconds() / 60
                breached = lag_min > sla.max_lag_minutes

            results.append(FreshnessResult(
                table=sla.name, last_updated=last_upd,
                lag_minutes=round(lag_min, 1),
                sla_minutes=sla.max_lag_minutes,
                breached=breached,
                severity=sla.severity if breached else 'OK'
            ))
        return results

    def report(self, results: list):
        icons = {'CRITICAL': '🔴', 'ERROR': '🟠', 'WARNING': '🟡', 'OK': '✅'}
        print(f'{'Table':<35} {'Lag(min)':>10} {'SLA(min)':>10} {'Status':>12}')
        print('-' * 72)
        for r in results:
            icon = icons.get(r.severity, '❓')
            lag_str = f'{r.lag_minutes:.1f}' if r.lag_minutes != float('inf') else 'UNKNOWN'
            print(f'{icon} {r.table:<33} {lag_str:>10} {r.sla_minutes:>10} {r.severity:>12}')


# Simulate table metadata
now = datetime.utcnow()
table_meta = {
    'hotel.prices.streaming':   now - timedelta(minutes=7),    # within SLA
    'booking_events':           now - timedelta(minutes=12),   # BREACHED (SLA=5)
    'financial_unified_daily':  now - timedelta(minutes=95),   # BREACHED (SLA=90)
    'hotel_features_daily':     now - timedelta(minutes=100),  # within SLA
    # ml_training_dataset missing — unknown
}

monitor = GoFreshMonitor()
results = monitor.check_all(table_meta)
monitor.report(results)

breached = [r for r in results if r.breached]
print(f'\n{len(breached)} SLA breaches detected. Paging on-call for CRITICAL breaches.')


---
### P15 ⭐⭐⭐ — Anomaly Detection on Pipeline Metrics

**Problem:** Given a time series of daily booking counts per hotel, detect anomalies
using both Z-score (simple) and IQR methods.
Explain when you'd use each, and what you'd do on detection.

⏱️ Target: **10 minutes**


In [ ]:
# ✅ SOLUTION P15 — Pipeline Anomaly Detection
import numpy as np, pandas as pd

np.random.seed(42)
# Simulate 90 days of daily booking counts for a hotel
normal_days = np.random.normal(100, 15, 90)
# Inject anomalies
normal_days[20] = 5    # sudden drop (data pipeline issue?)
normal_days[45] = 250  # spike (promotion? or duplicate records?)
normal_days[70] = 8    # another drop

dates = pd.date_range('2025-01-01', periods=90)
ts = pd.Series(normal_days.clip(0), index=dates, name='booking_count')

def zscore_anomaly(series, threshold=3.0):
    mean, std = series.mean(), series.std()
    zscores = (series - mean) / std
    return zscores.abs() > threshold, zscores

def iqr_anomaly(series, k=1.5):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - k*IQR, Q3 + k*IQR
    return (series < lo) | (series > hi), (lo, hi)

z_flags, zscores = zscore_anomaly(ts)
iqr_flags, (lo, hi) = iqr_anomaly(ts)

print('=== Anomalies Detected ===')
anomalies = pd.DataFrame({
    'booking_count': ts,
    'zscore':        zscores.round(2),
    'z_flag':        z_flags,
    'iqr_flag':      iqr_flags,
})
detected = anomalies[anomalies['z_flag'] | anomalies['iqr_flag']]
print(detected)

print(f'\nIQR bounds: [{lo:.1f}, {hi:.1f}]')
print(f'Z-score threshold: ±3.0 (covers 99.7% of normal)')

print()
print('=== When to use each? ===')
comparison = [
    ('Z-score', 'Data is roughly normal', 'Sensitive to extreme outliers affecting mean/std'),
    ('IQR',     'Skewed data, or unknown distribution', 'Less sensitive — misses moderate anomalies'),
    ('Prophet', 'Strong seasonality (weekly/holiday patterns)', 'Heavy dependency, slower to set up'),
]
for method, use_when, limitation in comparison:
    print(f'  {method:<10}: Use when {use_when}')
    print(f'             Limitation: {limitation}')
    print()

print('=== Response Playbook on Detection ===')
playbook = [
    'IMMEDIATELY: Check if upstream pipeline ran (GoFresh freshness check)',
    'IMMEDIATELY: Check Kafka consumer lag — did we stop consuming?',
    'INVESTIGATE: Is it real-world event (promotion, holiday) or data artifact?',
    'IF DATA ISSUE: Halt downstream pipelines (FINUDP pattern) before bad data propagates',
    'IF REAL EVENT: Log explanation, adjust anomaly detector thresholds temporarily',
    'ALWAYS: Write post-mortem. Add this pattern as a business-rule check for next time',
]
for i, step in enumerate(playbook, 1):
    print(f'  {i}. {step}')


---
## 🎤 System Design Communication (Live Session)


---
### P16 — Verbal Answer Templates (Read + Internalize)

These are the exact sentence patterns that signal seniority in a live session.

#### Opening any system design question
```
'Before I design, let me clarify a few things:
 What's the expected QPS and data volume?
 What's the acceptable latency — real-time, near-real-time, or batch?
 Who are the consumers — ML models, dashboards, or APIs?
 Do we need to support backfill or reprocessing?'

[After getting answers]
'OK, so let me restate the constraints: we're looking at X QPS,
 Y ms latency, with Z consumers, and we need eventual consistency.
 I'll start with the high-level architecture then drill into the hard parts.'
```

#### Justifying a technology choice
```
WEAK:  'I'll use Kafka.'

STRONG: 'I'll use Kafka here for three reasons:
         First, decoupling — producers and consumers evolve independently.
         Second, fan-out — the ML team and BI team can both consume
         the same topic without coupling their pipelines.
         Third, reprocessing — with 7-day retention, we can replay
         the topic if a downstream consumer has a bug.
         The trade-off is operational overhead of running Kafka clusters,
         but at Agoda's scale that cost is already absorbed.'
```

#### Presenting a trade-off
```
WEAK:  'I chose eventual consistency.'

STRONG: 'I chose eventual consistency here for two reasons.
         First, strong consistency would require distributed locks across
         our price cache, which would add 20-50ms latency per request
         and reduce throughput by ~40%.
         Second, a 5-minute stale price in search results is acceptable
         for the user experience — we lock the price at checkout anyway
         using a Quote ID with a 15-minute TTL.
         The only case where this breaks down is if a hotel goes from
         available to fully booked within that 5-minute window, which
         we handle with a final availability re-check at booking time.'
```

#### When you don't know something
```
WEAK:  [silence] or 'I'm not sure'

STRONG: 'I haven't worked directly with StarRocks in production,
         but based on what I know about columnar OLAP engines,
         my assumption would be X. Is that aligned with how it
         works in your system? I'm happy to adjust the design
         based on the actual behavior.'
```

#### Closing a design
```
'Let me summarize the key decisions:
 1. [Component] using [Technology] because [reason] with [trade-off acknowledged]
 2. [Component] using [Technology] because [reason] with [trade-off acknowledged]
 3. For monitoring: four golden signals on each hop, GoFresh for freshness SLAs,
    PagerDuty for CRITICAL alerts.
 4. If load 10x's, the first bottleneck is [X], which I'd address by [Y].
 Are there specific components you'd like me to go deeper on?'
```

---

### P17 — Timed Whiteboard: Design a Hotel Click-Stream Analytics Pipeline

**Set a 20-minute timer. Design this system out loud as if the interviewer is watching.**

**Requirements given:**
- Capture every user click/search/view/booking event from the Agoda website
- 50M events/day, spiky (10x during peak seasons)
- BI team needs: hourly dashboards, funnel analysis, per-destination trends
- ML team needs: session features for personalization model (< 1 hr freshness)
- Finance team needs: confirmed booking counts for revenue reconciliation

**Your design must cover:**
1. Ingestion layer (how events get from browser → system)
2. Kafka topic structure (how many topics, partitioning)
3. Processing (batch vs stream, which for whom)
4. Storage (what goes where)
5. At least 2 failure modes and their mitigations
6. Monitoring approach

**Reference answer is below — only look after you've attempted it.**


In [3]:
# ✅ P17 REFERENCE DESIGN — Clickstream Analytics Pipeline
reference_design = '''
=== CLARIFYING QUESTIONS ASKED ===
Q: Real-time latency for dashboards? A: hourly is fine
Q: Session definition? A: 30-min inactivity = new session
Q: Exactly-once needed? A: For booking events yes, for clicks best-effort OK

=== BACK-OF-ENVELOPE ===
50M events/day = 578 events/sec avg = ~5,780 peak (10x)
Event size: ~500 bytes avg
Throughput: 5,780 * 500 = 2.89 MB/sec peak -> Kafka handles this easily
Daily storage: 50M * 500B = 25GB/day -> ~9TB/year (use compression: ~2TB)

=== INGESTION ===
Browser/App -> API Gateway (event collector)
  - Agoda 2-Step Logging: app writes to local disk (non-blocking)
  - Forwarder sends to Kafka (handles retries independently)
  - For booking events: direct Kafka write (latency-critical)

=== KAFKA TOPIC STRUCTURE ===
user.clicks          -> partition by user_id, retention 7d, RF=2
user.searches        -> partition by destination, retention 7d, RF=2
hotel.bookings       -> partition by hotel_id, retention 30d, RF=3, exactly-once
sessions.raw         -> derived topic from clicks, partition by session_id

=== PROCESSING ===
STREAM (Spark Structured Streaming, 5-min micro-batch):
  Consumers: user.clicks + user.searches
  Jobs:
    1. Session stitcher: group events by user+30min window -> session features
    2. Real-time funnel: SEARCH->VIEW->BOOK counts per hour per destination
  Output: Redis (session features for ML) + staging table in StarRocks

BATCH (Spark, hourly trigger via Airflow):
  Consumers: all topics via HDFS
  Jobs:
    1. Full session reconstruction (accurate, handles late events)
    2. Revenue reconciliation (bookings -> confirmed revenue)
    3. ML training feature computation (with PIT join)
  Output: StarRocks (BI dashboards) + HDFS Parquet (ML feature store)

=== STORAGE ===
Redis:       session features (ML real-time serving, <10ms, TTL=1h)
StarRocks:   hourly/daily aggregations (BI dashboards, <1s queries)
HDFS:        raw events + processed features (ML training, long retention)
Elasticsearch: hotel search index (updated from bookings stream)

=== FAILURE MODES ===
1. Kafka lag spikes during Songkran peak:
   Detect: consumer lag > 500K for > 5min -> PagerDuty alert
   Mitigate: pre-scale Spark consumer replicas 48h before event
   Fallback: session stream drops non-critical clicks, processes bookings first

2. HDFS write fails mid-batch:
   Detect: Spark job fails, checkpoint not committed
   Mitigate: Spark restarts from last checkpoint automatically
   Data safety: Kafka retains 7d, so replay is always possible

=== MONITORING ===
Grafana: Kafka consumer lag, Spark job duration, Redis hit rate, HDFS usage
GoFresh: table freshness SLAs (hourly StarRocks update, daily HDFS partition)
Data Quality: event count anomaly detection (booking drop >40% = HALT + alert)
Business: funnel step drop alerts (if SEARCH->VIEW rate drops >20%: investigate)
'''
print(reference_design)



=== CLARIFYING QUESTIONS ASKED ===
Q: Real-time latency for dashboards? A: hourly is fine
Q: Session definition? A: 30-min inactivity = new session
Q: Exactly-once needed? A: For booking events yes, for clicks best-effort OK

=== BACK-OF-ENVELOPE ===
50M events/day = 578 events/sec avg = ~5,780 peak (10x)
Event size: ~500 bytes avg
Throughput: 5,780 * 500 = 2.89 MB/sec peak -> Kafka handles this easily
Daily storage: 50M * 500B = 25GB/day -> ~9TB/year (use compression: ~2TB)

=== INGESTION ===
Browser/App -> API Gateway (event collector)
  - Agoda 2-Step Logging: app writes to local disk (non-blocking)
  - Forwarder sends to Kafka (handles retries independently)
  - For booking events: direct Kafka write (latency-critical)

=== KAFKA TOPIC STRUCTURE ===
user.clicks          -> partition by user_id, retention 7d, RF=2
user.searches        -> partition by destination, retention 7d, RF=2
hotel.bookings       -> partition by hotel_id, retention 30d, RF=3, exactly-once
sessions.raw        

---
## 🚀 Live Coding Survival Checklist

Keep this open during practice sessions:

```
BEFORE YOU TYPE:
  □ Restate the problem in your own words
  □ State your assumptions out loud
  □ Outline your approach (1-2 sentences) before coding

WHILE CODING:
  □ Narrate non-obvious decisions as you type
  □ Name variables clearly (not x, y, z)
  □ Write in logical blocks, not one giant expression
  □ If stuck: say what you're thinking, not silent

AFTER CODING:
  □ Trace through a 2-3 row example before running
  □ State time complexity if relevant
  □ Proactively mention one improvement: 'In production I would also...'

FOR SPARK SPECIFICALLY:
  □ Always mention: shuffle.partitions tuning
  □ Always mention: AQE for skew handling
  □ Always mention: checkpoint location for streaming
  □ Always prefer parquet over CSV for output

FOR SQL SPECIFICALLY:
  □ Use CTEs for multi-step logic (never nested subqueries 3 levels deep)
  □ NULLIF() for division to prevent divide-by-zero
  □ State index assumption: 'assuming we have an index on hotel_id'

FOR SYSTEM DESIGN:
  □ Ask clarifying questions FIRST (2 minutes)
  □ Back-of-envelope SECOND (1 minute)
  □ High-level boxes THIRD
  □ Deep dive on 2 hardest components
  □ End with: failure modes + monitoring
```

---
> **Next step:** Ping me when you've worked through all 15 problems.
> We'll run a full 60-minute mock interview — I'll play the Agoda interviewer.
